# Stage 4: LLM-Based Biomedical Information Extraction

This notebook implements a Large Language Model (LLM)-based biomedical information extraction pipeline using the BioRED dataset.

The objective is to automatically identify biomedical entities and the semantic relations between them from biomedical literature using an open-source instruction-tuned LLM. The extracted information is validated and evaluated against the BioRED gold-standard annotations before being used for downstream knowledge graph construction.

The implementation follows a structured workflow consisting of model preparation, prompt engineering, entity extraction, relation extraction, validation, and quantitative evaluation.


## Pipeline Overview

### 4.1 Environment Setup and Data Loading
- 4.1A Mount Google Drive
- 4.1B Import Required Libraries
- 4.1C Load Prepared BioRED Documents
- 4.1D Create Pilot and Evaluation Splits

### 4.2 LLM Initialisation
- 4.2A Configure Model Parameters
- 4.2B Load Tokenizer
- 4.2C Load Quantised Qwen Model
- 4.2D Verify Model Configuration

### 4.3 Biomedical Extraction Schema
- 4.3A Define BioRED Entity Types
- 4.3B Define BioRED Relation Types
- 4.3C Define Entity JSON Schema with Character Offsets
- 4.3D Define Relation JSON Schema
- 4.3E Define Validation Rules

### 4.4 Biomedical Entity Extraction
- 4.4A Construct the Entity-Only Few-Shot Prompt
- 4.4B Generate Entity Predictions
- 4.4C Parse JSON Output
- 4.4D Validate Predicted Entity Types and IDs
- 4.4E Validate Text and Character Offsets
- 4.4F Evaluate Entity Extraction Performance
- 4.4G Freeze the Entity Prompt

### 4.5 Relation Extraction
- 4.5A Prepare the Validated Entity List
- 4.5B Construct the Relation-Only Prompt
- 4.5C Generate Relation Predictions
- 4.5D Parse JSON Output
- 4.5E Validate Relation Types and Entity References
- 4.5F Evaluate Relation Extraction Performance
- 4.5G Freeze the Relation Prompt

### 4.6 Large-Scale Development Set Evaluation
- 4.6A Apply the Frozen Pipeline to the Development Set
- 4.6B Save Predictions After Every Document
- 4.6C Resume from Incomplete Documents
- 4.6D Compute Overall NER and RE Metrics

### 4.7 Error Analysis
- 4.7A Analyse Entity Span Errors
- 4.7B Analyse Entity Type Errors
- 4.7C Analyse Missed and Repeated Mentions
- 4.7D Analyse Relation Errors
- 4.7E Identify Common Failure Cases

### 4.8 Final Model Comparison
- 4.8A Compare Qwen with Fine-Tuned PubMedBERT
- 4.8B Compare Accuracy, Runtime and Cost
- 4.8C Summarise Strengths and Limitations

### 4.9 Stage Summary

## Stage 4.1: Environment Setup and Data Loading

This stage prepares the Google Colab environment and loads the BioRED data required for the two-stage LLM extraction pipeline.

The development data will later be divided into a small prompt-development sample and a separate evaluation set. Entity extraction and relation extraction will be performed independently.

In [3]:
# Stage 4.1A: Mount Google Drive

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
# Stage 4.1B: Import Required Libraries

import json
import pickle
import random
import re

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

In [5]:
# Set random seed for reproducibility

random.seed(42)
torch.manual_seed(42)

In [6]:
# Stage 4.1C: Load Prepared BioRED Documents

DATASET_PATH = "/content/drive/MyDrive/Capstone_Project/BioRED_Dataset/"

TRAIN_FILE = DATASET_PATH + "Train.BioC.JSON"
DEV_FILE = DATASET_PATH + "Dev.BioC.JSON"
TEST_FILE = DATASET_PATH + "Test.BioC.JSON"

with open(TRAIN_FILE, "r", encoding="utf-8") as file:
    train_data = json.load(file)

with open(DEV_FILE, "r", encoding="utf-8") as file:
    dev_data = json.load(file)

with open(TEST_FILE, "r", encoding="utf-8") as file:
    test_data = json.load(file)

print("BioRED dataset loaded successfully.")

BioRED dataset loaded successfully.


In [8]:
# Stage 4.1D: Inspect One BioRED Annotation

sample_annotation = (
    train_data["documents"][0]["passages"][0]["annotations"][0]
)

print(sample_annotation)
print()
print(sample_annotation.keys())
print("\n")

sample_relation = train_data["documents"][0]["relations"][0]

print(sample_relation)
print()
print(sample_relation.keys())

{'id': '0', 'infons': {'identifier': '3175', 'type': 'GeneOrGeneProduct'}, 'text': 'Hepatocyte nuclear factor-6', 'locations': [{'offset': 0, 'length': 27}]}

dict_keys(['id', 'infons', 'text', 'locations'])


{'id': 'R0', 'infons': {'entity1': '3175', 'entity2': 'D003924', 'type': 'Association', 'novel': 'No'}}

dict_keys(['id', 'infons'])


In [9]:
# Stage 4.1E: Inspect Passage Structure and Offsets

sample_document = train_data["documents"][0]

print("Document ID:", sample_document["id"])
print("Number of passages:", len(sample_document["passages"]))
print()

for index, passage in enumerate(sample_document["passages"]):
    print(
        "Passage:",
        index,
        "| Offset:",
        passage["offset"],
        "| Length:",
        len(passage["text"])
    )
    print(repr(passage["text"][:100]))
    print()

Document ID: 10491763
Number of passages: 2

Passage: 0 | Offset: 0 | Length: 158
'Hepatocyte nuclear factor-6: associations between genetic variability and type II diabetes and betwe'

Passage: 1 | Offset: 159 | Length: 1638
'The transcription factor hepatocyte nuclear factor (HNF)-6 is an upstream regulator of several genes'



In [10]:
# Stage 4.1F: Reconstruct One BioRED Document

def reconstruct_document_text(document):
    passages = document["passages"]

    document_length = max(
        passage["offset"] + len(passage["text"])
        for passage in passages
    )

    characters = [" "] * document_length

    for passage in passages:
        start = passage["offset"]
        end = start + len(passage["text"])
        characters[start:end] = passage["text"]

    return "".join(characters)


sample_text = reconstruct_document_text(sample_document)

print("Reconstructed text length:", len(sample_text))
print("Expected length:", 159 + 1638)
print()
print(repr(sample_text[145:175]))

Reconstructed text length: 1797
Expected length: 1797

'in secretion. The transcriptio'


In [11]:
# Stage 4.1G: Prepare One BioRED Document

def prepare_biored_document(document):
    text = reconstruct_document_text(document)

    gold_entities = []

    for passage in document["passages"]:
        for annotation in passage.get("annotations", []):
            infons = annotation.get("infons", {})

            for location in annotation.get("locations", []):
                start = location["offset"]
                end = start + location["length"]

                gold_entities.append({
                    "annotation_id": annotation["id"],
                    "identifier": infons.get("identifier"),
                    "type": infons.get("type"),
                    "text": annotation["text"],
                    "start": start,
                    "end": end
                })

    gold_relations = []

    for relation in document.get("relations", []):
        infons = relation.get("infons", {})

        gold_relations.append({
            "relation_id": relation["id"],
            "type": infons.get("type"),
            "entity1_identifier": infons.get("entity1"),
            "entity2_identifier": infons.get("entity2"),
            "novel": infons.get("novel")
        })

    return {
        "document_id": str(document["id"]),
        "text": text,
        "gold_entities": gold_entities,
        "gold_relations": gold_relations
    }


prepared_sample_document = prepare_biored_document(
    sample_document
)

print("Document ID:", prepared_sample_document["document_id"])
print("Text length:", len(prepared_sample_document["text"]))
print("Gold entities:", len(prepared_sample_document["gold_entities"]))
print("Gold relations:", len(prepared_sample_document["gold_relations"]))

print("\nFirst entity:")
print(prepared_sample_document["gold_entities"][0])

print("\nFirst relation:")
print(prepared_sample_document["gold_relations"][0])

Document ID: 10491763
Text length: 1797
Gold entities: 32
Gold relations: 3

First entity:
{'annotation_id': '0', 'identifier': '3175', 'type': 'GeneOrGeneProduct', 'text': 'Hepatocyte nuclear factor-6', 'start': 0, 'end': 27}

First relation:
{'relation_id': 'R0', 'type': 'Association', 'entity1_identifier': '3175', 'entity2_identifier': 'D003924', 'novel': 'No'}


In [12]:
# Stage 4.1H: Prepare All BioRED Documents

prepared_train_documents = [
    prepare_biored_document(document)
    for document in train_data["documents"]
]

prepared_dev_documents = [
    prepare_biored_document(document)
    for document in dev_data["documents"]
]

prepared_test_documents = [
    prepare_biored_document(document)
    for document in test_data["documents"]
]

print("Prepared training documents:", len(prepared_train_documents))
print("Prepared development documents:", len(prepared_dev_documents))
print("Prepared test documents:", len(prepared_test_documents))

Prepared training documents: 400
Prepared development documents: 100
Prepared test documents: 100


In [13]:
# Stage 4.1I: Validate Gold Entity Offsets

def validate_gold_entity_offsets(documents):
    errors = []

    for document in documents:
        text = document["text"]

        for entity in document["gold_entities"]:
            extracted_text = text[
                entity["start"]:entity["end"]
            ]

            if extracted_text != entity["text"]:
                errors.append({
                    "document_id": document["document_id"],
                    "annotation_id": entity["annotation_id"],
                    "expected": entity["text"],
                    "extracted": extracted_text,
                    "start": entity["start"],
                    "end": entity["end"]
                })

    return errors


train_offset_errors = validate_gold_entity_offsets(
    prepared_train_documents
)

dev_offset_errors = validate_gold_entity_offsets(
    prepared_dev_documents
)

test_offset_errors = validate_gold_entity_offsets(
    prepared_test_documents
)

print("Training offset errors:", len(train_offset_errors))
print("Development offset errors:", len(dev_offset_errors))
print("Test offset errors:", len(test_offset_errors))

Training offset errors: 0
Development offset errors: 0
Test offset errors: 0


In [14]:
# Stage 4.1J: Validate Gold Relation References

def split_biored_identifiers(identifier):
    if identifier is None:
        return set()

    return {
        part.strip()
        for part in str(identifier).split(",")
        if part.strip()
    }


def validate_gold_relation_references(documents):
    errors = []

    for document in documents:
        valid_identifiers = set()

        for entity in document["gold_entities"]:
            valid_identifiers.update(
                split_biored_identifiers(
                    entity.get("identifier")
                )
            )

        for relation in document["gold_relations"]:
            for endpoint in [
                "entity1_identifier",
                "entity2_identifier"
            ]:
                identifier = relation.get(endpoint)

                if identifier not in valid_identifiers:
                    errors.append({
                        "document_id": document["document_id"],
                        "relation_id": relation["relation_id"],
                        "endpoint": endpoint,
                        "identifier": identifier
                    })

    return errors


train_relation_errors = validate_gold_relation_references(
    prepared_train_documents
)

dev_relation_errors = validate_gold_relation_references(
    prepared_dev_documents
)

test_relation_errors = validate_gold_relation_references(
    prepared_test_documents
)

print("Training relation reference errors:", len(train_relation_errors))
print("Development relation reference errors:", len(dev_relation_errors))
print("Test relation reference errors:", len(test_relation_errors))

Training relation reference errors: 0
Development relation reference errors: 0
Test relation reference errors: 0


In [15]:
# Stage 4.1K: Save Prepared BioRED Documents

from pathlib import Path

OUTPUT_PATH = Path(
    "/content/drive/MyDrive/Capstone_Project/Stage_4_Outputs"
)

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

prepared_data = {
    "train": prepared_train_documents,
    "development": prepared_dev_documents,
    "test": prepared_test_documents
}

prepared_data_file = OUTPUT_PATH / "prepared_biored_documents.pkl"

with open(prepared_data_file, "wb") as file:
    pickle.dump(prepared_data, file)

print("Prepared BioRED documents saved successfully.")
print("Saved to:", prepared_data_file)

Prepared BioRED documents saved successfully.
Saved to: /content/drive/MyDrive/Capstone_Project/Stage_4_Outputs/prepared_biored_documents.pkl


In [16]:
# Stage 4.1L: Verify Saved Prepared Data

with open(prepared_data_file, "rb") as file:
    reloaded_prepared_data = pickle.load(file)

print(
    "Reloaded training documents:",
    len(reloaded_prepared_data["train"])
)
print(
    "Reloaded development documents:",
    len(reloaded_prepared_data["development"])
)
print(
    "Reloaded test documents:",
    len(reloaded_prepared_data["test"])
)

Reloaded training documents: 400
Reloaded development documents: 100
Reloaded test documents: 100


In [17]:
# Stage 4.2A: Verify GPU Availability

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Change the Colab runtime to T4 GPU.")

CUDA available: True
GPU: NVIDIA L4


In [18]:
# Stage 4.2B: Load Tokenizer

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

print("Tokenizer loaded successfully.")
print("Model:", MODEL_NAME)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokenizer loaded successfully.
Model: Qwen/Qwen2.5-7B-Instruct


In [19]:
# Stage 4.2C: Install Quantisation Dependencies

!pip install -q --upgrade bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 67.5 MB/s eta 0:00:00


In [20]:
# Stage 4.2C: Load Quantised LLM

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16
)

print("Qwen loaded successfully.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen loaded successfully.


In [21]:
# Stage 4.2D: Verify Model Configuration

print("Model name:", MODEL_NAME)
print("Model class:", model.__class__.__name__)
print("Training mode:", model.training)
print("4-bit model:", getattr(model, "is_loaded_in_4bit", False))

model_device = next(model.parameters()).device
print("Model device:", model_device)

allocated_memory_gb = torch.cuda.memory_allocated() / (1024 ** 3)
reserved_memory_gb = torch.cuda.memory_reserved() / (1024 ** 3)

print(f"Allocated GPU memory: {allocated_memory_gb:.2f} GB")
print(f"Reserved GPU memory: {reserved_memory_gb:.2f} GB")

Model name: Qwen/Qwen2.5-7B-Instruct
Model class: Qwen2ForCausalLM
Training mode: False
4-bit model: True
Model device: cuda:0
Allocated GPU memory: 5.18 GB
Reserved GPU memory: 5.33 GB


In [22]:
# Stage 4.3A: Define BioRED Entity Types

BIORED_ENTITY_TYPES = [
    "CellLine",
    "ChemicalEntity",
    "DiseaseOrPhenotypicFeature",
    "GeneOrGeneProduct",
    "OrganismTaxon",
    "SequenceVariant"
]

print("Number of BioRED entity types:", len(BIORED_ENTITY_TYPES))

for entity_type in BIORED_ENTITY_TYPES:
    print("-", entity_type)

Number of BioRED entity types: 6
- CellLine
- ChemicalEntity
- DiseaseOrPhenotypicFeature
- GeneOrGeneProduct
- OrganismTaxon
- SequenceVariant


In [23]:
# Stage 4.3B: Define BioRED Relation Types

BIORED_RELATION_TYPES = [
    "Association",
    "Bind",
    "Comparison",
    "Conversion",
    "Cotreatment",
    "Drug_Interaction",
    "Negative_Correlation",
    "Positive_Correlation"
]

print("Number of BioRED relation types:", len(BIORED_RELATION_TYPES))

for relation_type in BIORED_RELATION_TYPES:
    print("-", relation_type)

Number of BioRED relation types: 8
- Association
- Bind
- Comparison
- Conversion
- Cotreatment
- Drug_Interaction
- Negative_Correlation
- Positive_Correlation


In [24]:
# Stage 4.3C: Define JSON Output Schemas

ENTITY_OUTPUT_SCHEMA = {
    "entities": [
        {
            "id": "E1",
            "text": "exact entity text from the document",
            "type": "one valid BioRED entity type"
        }
    ]
}

RELATION_OUTPUT_SCHEMA = {
    "relations": [
        {
            "head": "E1",
            "tail": "E2",
            "type": "one valid BioRED relation type"
        }
    ]
}

print("Entity output schema:")
print(json.dumps(ENTITY_OUTPUT_SCHEMA, indent=2))

print("\nRelation output schema:")
print(json.dumps(RELATION_OUTPUT_SCHEMA, indent=2))

Entity output schema:
{
  "entities": [
    {
      "id": "E1",
      "text": "exact entity text from the document",
      "type": "one valid BioRED entity type"
    }
  ]
}

Relation output schema:
{
  "relations": [
    {
      "head": "E1",
      "tail": "E2",
      "type": "one valid BioRED relation type"
    }
  ]
}


In [25]:
# Stage 4.3D: Define Validation Rules

VALIDATION_RULES = {
    "entity_rules": [
        "Entity text must appear exactly in the document.",
        "Entity type must be one of the BioRED entity types.",
        "Each entity must have a unique identifier.",
        "Character offsets must exactly match the entity text.",
        "Entity start offset must be smaller than the end offset."
    ],
    "relation_rules": [
        "Relation type must be one of the BioRED relation types.",
        "Head and tail entities must exist in the predicted entity list.",
        "Head and tail entity identifiers must be different.",
        "Duplicate relations are not allowed."
    ]
}

print("Entity Validation Rules:")
for rule in VALIDATION_RULES["entity_rules"]:
    print("-", rule)

print("\nRelation Validation Rules:")
for rule in VALIDATION_RULES["relation_rules"]:
    print("-", rule)

Entity Validation Rules:
- Entity text must appear exactly in the document.
- Entity type must be one of the BioRED entity types.
- Each entity must have a unique identifier.
- Character offsets must exactly match the entity text.
- Entity start offset must be smaller than the end offset.

Relation Validation Rules:
- Relation type must be one of the BioRED relation types.
- Head and tail entities must exist in the predicted entity list.
- Head and tail entity identifiers must be different.
- Duplicate relations are not allowed.


## Stage 4.4: Biomedical Entity Extraction

This stage performs biomedical entity extraction independently from relation extraction.

The LLM will be prompted to identify only explicitly mentioned BioRED entities and return each entity with its exact document text, permitted entity type, and character offsets. The generated output will then be parsed, validated, and evaluated against the BioRED gold-standard entity annotations before relation extraction begins.

In [26]:
# Stage 4.4A: Select Pilot Documents for Few-Shot Evaluation

PILOT_SIZE = 5
PILOT_SEED = 42

random.seed(PILOT_SEED)

pilot_documents = random.sample(
    prepared_dev_documents,
    PILOT_SIZE
)

print("Number of pilot documents:", len(pilot_documents))
print("Pilot document IDs:")

for document in pilot_documents:
    print("-", document["document_id"])

Number of pilot documents: 5
Pilot document IDs:
- 16680561
- 27840894
- 16277682
- 20195852
- 21130517


In [27]:
# Stage 4.4B: Parse JSON Responses

import json
import re


def parse_json_response(response_text):
    cleaned_text = response_text.strip()

    # Prefer JSON inside a fenced code block.
    fenced_match = re.search(
        r"```(?:json)?\s*(\{.*?\})\s*```",
        cleaned_text,
        flags=re.IGNORECASE | re.DOTALL
    )

    if fenced_match:
        json_text = fenced_match.group(1)
    else:
        # Otherwise extract from the first opening brace
        # to the final closing brace.
        start = cleaned_text.find("{")
        end = cleaned_text.rfind("}")

        if start == -1 or end == -1 or end < start:
            return None, "No complete JSON object was found."

        json_text = cleaned_text[start:end + 1]

    try:
        return json.loads(json_text), None
    except json.JSONDecodeError as error:
        return None, str(error)

In [28]:
# Stage 4.4C: Validate Predicted Entity Surface Forms

def validate_entity_surface_forms(
    parsed_output,
    document_text,
    permitted_entity_types
):
    errors = []
    warnings = []
    valid_entities = []

    if not isinstance(parsed_output, dict):
        return {
            "valid": False,
            "entities": [],
            "errors": ["The parsed output must be a JSON object."],
            "warnings": []
        }

    entities = parsed_output.get("entities")

    if not isinstance(entities, list):
        return {
            "valid": False,
            "entities": [],
            "errors": ["The 'entities' field must be a list."],
            "warnings": []
        }

    seen_ids = set()
    seen_surface_type_pairs = set()

    for index, entity in enumerate(entities):
        label = f"Entity at index {index}"
        entity_errors = []

        if not isinstance(entity, dict):
            errors.append(f"{label}: must be a JSON object.")
            continue

        entity_id = entity.get("id")
        entity_text = entity.get("text")
        entity_type = entity.get("type")

        if not isinstance(entity_id, str) or not entity_id.strip():
            entity_errors.append(f"{label}: missing or invalid 'id'.")
        elif entity_id in seen_ids:
            entity_errors.append(
                f"{label}: duplicate ID {entity_id!r}."
            )

        if not isinstance(entity_text, str) or not entity_text:
            entity_errors.append(
                f"{label}: missing or invalid 'text'."
            )

        if entity_type not in permitted_entity_types:
            entity_errors.append(
                f"{label}: invalid entity type {entity_type!r}."
            )

        pair = (entity_text, entity_type)

        if pair in seen_surface_type_pairs:
            entity_errors.append(
                f"{label}: duplicate text/type combination {pair!r}."
            )

        occurrence_count = (
            document_text.count(entity_text)
            if isinstance(entity_text, str)
            else 0
        )

        if occurrence_count == 0:
            entity_errors.append(
                f"{label}: text not found exactly in the document: "
                f"{entity_text!r}."
            )

        if entity_errors:
            errors.extend(entity_errors)
            continue

        seen_ids.add(entity_id)
        seen_surface_type_pairs.add(pair)

        if occurrence_count > 1:
            warnings.append(
                f"{label}: {entity_text!r} occurs "
                f"{occurrence_count} times and will be expanded "
                "into separate mention-level entities."
            )

        valid_entities.append({
            "id": entity_id,
            "text": entity_text,
            "type": entity_type,
            "occurrence_count": occurrence_count
        })

    return {
        "valid": len(errors) == 0,
        "entities": valid_entities,
        "errors": errors,
        "warnings": warnings
    }

In [29]:
# Stage 4.4D: Expand Surface Forms into Mention-Level Entities

def expand_surface_forms_to_mentions(
    validated_entities,
    document_text
):
    mention_entities = []

    for entity in validated_entities:
        entity_text = entity["text"]
        entity_type = entity["type"]
        source_id = entity["id"]

        start_position = 0
        occurrence_number = 1

        while True:
            start = document_text.find(
                entity_text,
                start_position
            )

            if start == -1:
                break

            end = start + len(entity_text)

            mention_entities.append({
                "id": f"{source_id}_M{occurrence_number}",
                "source_surface_id": source_id,
                "text": entity_text,
                "type": entity_type,
                "start": start,
                "end": end
            })

            occurrence_number += 1
            start_position = end

    mention_entities.sort(
        key=lambda entity: (
            entity["start"],
            entity["end"],
            entity["type"]
        )
    )

    return mention_entities


In [30]:
# Stage 4.4E: Validate Recovered Mention Offsets

def validate_recovered_mentions(
    mention_entities,
    document_text
):
    errors = []

    for entity in mention_entities:
        start = entity["start"]
        end = entity["end"]
        entity_text = entity["text"]

        if not isinstance(start, int) or not isinstance(end, int):
            errors.append({
                "id": entity["id"],
                "error": "Offsets must be integers."
            })
            continue

        if start < 0 or end > len(document_text) or start >= end:
            errors.append({
                "id": entity["id"],
                "error": "Invalid offset range.",
                "start": start,
                "end": end
            })
            continue

        extracted_text = document_text[start:end]

        if extracted_text != entity_text:
            errors.append({
                "id": entity["id"],
                "expected": entity_text,
                "extracted": extracted_text,
                "start": start,
                "end": end
            })

    return errors



## Stage 4.5: Few-Shot Biomedical Named Entity Recognition

This stage performs biomedical named entity recognition using few-shot prompting. Representative BioRED training documents and their gold entity annotations are included as demonstrations. The existing JSON parsing, validation, mention expansion, offset recovery, and strict evaluation functions are reused.

In [31]:
# Stage 4.5A: Select Few-Shot Training Examples

def select_few_shot_examples(
    training_documents,
    number_of_examples=3,
    minimum_entity_types=3
):
    eligible_documents = []

    for document in training_documents:
        entity_types = {
            entity["type"]
            for entity in document["gold_entities"]
        }

        if (
            len(entity_types) >= minimum_entity_types
            and len(document["gold_entities"]) > 0
        ):
            eligible_documents.append(document)

    eligible_documents.sort(
        key=lambda document: len(document["text"])
    )

    if len(eligible_documents) < number_of_examples:
        raise ValueError(
            "Not enough eligible training documents were found."
        )

    return eligible_documents[:number_of_examples]


few_shot_examples = select_few_shot_examples(
    training_documents=prepared_train_documents,
    number_of_examples=3
)

print("Few-shot examples selected:")

for document in few_shot_examples:
    entity_types = sorted({
        entity["type"]
        for entity in document["gold_entities"]
    })

    print(
        "- Document:",
        document["document_id"],
        "| Characters:",
        len(document["text"]),
        "| Gold entities:",
        len(document["gold_entities"]),
        "| Entity types:",
        entity_types
    )

Few-shot examples selected:
- Document: 7468724 | Characters: 355 | Gold entities: 8 | Entity types: ['ChemicalEntity', 'DiseaseOrPhenotypicFeature', 'OrganismTaxon']
- Document: 15602202 | Characters: 471 | Gold entities: 9 | Entity types: ['ChemicalEntity', 'DiseaseOrPhenotypicFeature', 'OrganismTaxon']
- Document: 18262054 | Characters: 501 | Gold entities: 9 | Entity types: ['DiseaseOrPhenotypicFeature', 'GeneOrGeneProduct', 'OrganismTaxon', 'SequenceVariant']


In [32]:
# Stage 4.5B: Convert Gold Entities to Compact Demonstration JSON

def gold_entities_to_demo_json(document):
    grouped_entities = {
        entity_type: []
        for entity_type in BIORED_ENTITY_TYPES
    }

    seen_pairs = set()

    for entity in sorted(
        document["gold_entities"],
        key=lambda item: (
            item["start"],
            item["end"],
            item["type"]
        )
    ):
        pair = (entity["text"], entity["type"])

        if pair in seen_pairs:
            continue

        seen_pairs.add(pair)
        grouped_entities[entity["type"]].append(
            entity["text"]
        )

    return {
        entity_type: entity_texts
        for entity_type, entity_texts
        in grouped_entities.items()
        if entity_texts
    }


few_shot_demo_outputs = [
    gold_entities_to_demo_json(document)
    for document in few_shot_examples
]

In [33]:
# Stage 4.5C: Build Compact Few-Shot Entity Extraction Prompt

def build_few_shot_entity_prompt(
    document,
    example_documents,
    example_outputs
):
    demonstration_blocks = []

    for index, (example_document, example_output) in enumerate(
        zip(example_documents, example_outputs),
        start=1
    ):
        demonstration_blocks.append(
            f"""
Example {index}

Document:
{example_document["text"]}

Output:
{json.dumps(example_output, ensure_ascii=False)}
""".strip()
        )

    demonstrations = "\n\n".join(demonstration_blocks)

    return f"""
You are a biomedical named entity recognition system.

Extract all distinct biomedical entity surface forms from the target document.

Allowed BioRED entity types:
{json.dumps(BIORED_ENTITY_TYPES, ensure_ascii=False)}

Rules:
1. Copy every entity exactly from the target document.
2. Return each distinct text-and-type combination only once.
3. Keep abbreviations and full names separate.
4. Use only the permitted entity types.
5. Omit entity types for which no entities are found.
6. Return only valid JSON using double quotation marks.

Follow these examples:

{demonstrations}

Target document:
{document["text"]}

Return a JSON object mapping each predicted entity type to a list of
distinct entity surface forms.
""".strip()

In [34]:
few_shot_prompt_preview = build_few_shot_entity_prompt(
    document=pilot_documents[0],
    example_documents=few_shot_examples,
    example_outputs=few_shot_demo_outputs
)

preview_token_count = len(
    tokenizer(
        few_shot_prompt_preview,
        add_special_tokens=False
    )["input_ids"]
)

print("Prompt characters:", len(few_shot_prompt_preview))
print("Prompt tokens:", preview_token_count)

print("\nPrompt preview:\n")
print(few_shot_prompt_preview[:3000])

Prompt characters: 4325
Prompt tokens: 1049

Prompt preview:

You are a biomedical named entity recognition system.

Extract all distinct biomedical entity surface forms from the target document.

Allowed BioRED entity types:
["CellLine", "ChemicalEntity", "DiseaseOrPhenotypicFeature", "GeneOrGeneProduct", "OrganismTaxon", "SequenceVariant"]

Rules:
1. Copy every entity exactly from the target document.
2. Return each distinct text-and-type combination only once.
3. Keep abbreviations and full names separate.
4. Use only the permitted entity types.
5. Omit entity types for which no entities are found.
6. Return only valid JSON using double quotation marks.

Follow these examples:

Example 1

Document:
Cardiovascular complications associated with terbutaline treatment for preterm labor. Severe cardiovascular complications occurred in eight of 160 patients treated with terbutaline for preterm labor. Associated corticosteroid therapy and twin gestations appear to be predisposing factors. 

In [35]:
# Stage 4.5D: Generate Compact Few-Shot Entity Predictions

def generate_few_shot_entity_predictions(
    document,
    example_documents,
    example_outputs,
    max_new_tokens=500
):
    prompt = build_few_shot_entity_prompt(
        document=document,
        example_documents=example_documents,
        example_outputs=example_outputs
    )

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    )

    model_device = next(model.parameters()).device

    model_inputs = {
        key: value.to(model_device)
        for key, value in model_inputs.items()
    }

    with torch.inference_mode():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_only = generated_ids[
        :,
        model_inputs["input_ids"].shape[1]:
    ]

    raw_response = tokenizer.decode(
        generated_only[0],
        skip_special_tokens=True
    ).strip()

    return {
        "document_id": document["document_id"],
        "input_token_count": model_inputs["input_ids"].shape[1],
        "generated_token_count": generated_only.shape[1],
        "reached_token_limit": (
            generated_only.shape[1] >= max_new_tokens
        ),
        "raw_response": raw_response
    }

In [36]:
# Stage 4.5E: Convert Grouped Predictions to Standard Entity JSON

def grouped_output_to_entity_json(grouped_output):
    entities = []
    seen_pairs = set()

    if not isinstance(grouped_output, dict):
        return {"entities": []}

    for entity_type in BIORED_ENTITY_TYPES:
        surface_forms = grouped_output.get(
            entity_type,
            []
        )

        if not isinstance(surface_forms, list):
            continue

        for entity_text in surface_forms:
            if not isinstance(entity_text, str):
                continue

            pair = (entity_text, entity_type)

            if pair in seen_pairs:
                continue

            seen_pairs.add(pair)

            entities.append({
                "id": f"E{len(entities) + 1}",
                "text": entity_text,
                "type": entity_type
            })

    return {
        "entities": entities
    }

In [ ]:
# Stage 4.5F: Run First Few-Shot Pilot Prediction

few_shot_pilot_output_1 = generate_few_shot_entity_predictions(
    document=pilot_documents[0],
    example_documents=few_shot_examples,
    example_outputs=few_shot_demo_outputs,
    max_new_tokens=500
)

print("Document ID:", few_shot_pilot_output_1["document_id"])
print("Input tokens:", few_shot_pilot_output_1["input_token_count"])
print("Generated tokens:", few_shot_pilot_output_1["generated_token_count"])
print(
    "Reached token limit:",
    few_shot_pilot_output_1["reached_token_limit"]
)

print("\nLLM response:\n")
print(few_shot_pilot_output_1["raw_response"])

Document ID: 16680561
Input tokens: 1078
Generated tokens: 88
Reached token limit: False

LLM response:

```json
{
  "ChemicalEntity": ["desipramine HCl", "cinacalcet HCl", "desipramine", "cinacalcet"],
  "DiseaseOrPhenotypicFeature": ["adverse events", "narrow therapeutic index"],
  "GeneOrGeneProduct": ["CYP2D6"],
  "OrganismTaxon": ["healthy subjects"]
}
```


In [ ]:
# Stage 4.5G: Parse and Convert the First Few-Shot Prediction

grouped_output_1, parsing_error_1 = parse_json_response(
    few_shot_pilot_output_1["raw_response"]
)

print("Valid grouped JSON:", parsing_error_1 is None)

if parsing_error_1 is None:
    parsed_few_shot_output_1 = grouped_output_to_entity_json(
        grouped_output_1
    )

    print(
        "Predicted entity surface forms:",
        len(parsed_few_shot_output_1["entities"])
    )

    print("\nStandard entity JSON:\n")
    print(json.dumps(
        parsed_few_shot_output_1,
        indent=2,
        ensure_ascii=False
    ))
else:
    print("Parsing error:", parsing_error_1)

Valid grouped JSON: True
Predicted entity surface forms: 8

Standard entity JSON:

{
  "entities": [
    {
      "id": "E1",
      "text": "desipramine HCl",
      "type": "ChemicalEntity"
    },
    {
      "id": "E2",
      "text": "cinacalcet HCl",
      "type": "ChemicalEntity"
    },
    {
      "id": "E3",
      "text": "desipramine",
      "type": "ChemicalEntity"
    },
    {
      "id": "E4",
      "text": "cinacalcet",
      "type": "ChemicalEntity"
    },
    {
      "id": "E5",
      "text": "adverse events",
      "type": "DiseaseOrPhenotypicFeature"
    },
    {
      "id": "E6",
      "text": "narrow therapeutic index",
      "type": "DiseaseOrPhenotypicFeature"
    },
    {
      "id": "E7",
      "text": "CYP2D6",
      "type": "GeneOrGeneProduct"
    },
    {
      "id": "E8",
      "text": "healthy subjects",
      "type": "OrganismTaxon"
    }
  ]
}


In [37]:
# Stage 4.5H: Define Strict NER Evaluation Function

def evaluate_entity_predictions(
    parsed_output,
    document
):
    validation = validate_entity_surface_forms(
        parsed_output=parsed_output,
        document_text=document["text"],
        permitted_entity_types=BIORED_ENTITY_TYPES
    )

    predicted_mentions = expand_surface_forms_to_mentions(
        validated_entities=validation["entities"],
        document_text=document["text"]
    )

    offset_errors = validate_recovered_mentions(
        mention_entities=predicted_mentions,
        document_text=document["text"]
    )

    gold_keys = {
        (entity["start"], entity["end"], entity["type"])
        for entity in document["gold_entities"]
    }

    predicted_keys = {
        (entity["start"], entity["end"], entity["type"])
        for entity in predicted_mentions
    }

    tp = len(gold_keys & predicted_keys)
    fp = len(predicted_keys - gold_keys)
    fn = len(gold_keys - predicted_keys)

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall
        else 0.0
    )

    return {
        "validation": validation,
        "predicted_mentions": predicted_mentions,
        "offset_errors": offset_errors,
        "gold_entities": len(gold_keys),
        "predicted_entities": len(predicted_keys),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [ ]:
# Stage 4.5I: Evaluate First Few-Shot Pilot

few_shot_result_1 = evaluate_entity_predictions(
    parsed_output=parsed_few_shot_output_1,
    document=pilot_documents[0]
)

print("Few-shot Pilot 1 strict NER evaluation")
print("---------------------------------------")
print("Gold entities:", few_shot_result_1["gold_entities"])
print("Predicted entities:", few_shot_result_1["predicted_entities"])
print("True positives:", few_shot_result_1["tp"])
print("False positives:", few_shot_result_1["fp"])
print("False negatives:", few_shot_result_1["fn"])
print(f"Precision: {few_shot_result_1['precision']:.4f}")
print(f"Recall:    {few_shot_result_1['recall']:.4f}")
print(f"F1-score:  {few_shot_result_1['f1']:.4f}")

Few-shot Pilot 1 strict NER evaluation
---------------------------------------
Gold entities: 28
Predicted entities: 29
True positives: 24
False positives: 5
False negatives: 4
Precision: 0.8276
Recall:    0.8571
F1-score:  0.8421


In [38]:
# Stage 4.5I: Evaluate All Pilot Documents

def evaluate_pilot_documents(
    pilot_documents,
    example_documents,
    example_outputs
):
    results = []

    for index, document in enumerate(pilot_documents, start=1):

        prediction = generate_few_shot_entity_predictions(
            document=document,
            example_documents=example_documents,
            example_outputs=example_outputs,
            max_new_tokens=500
        )

        grouped_output, parsing_error = parse_json_response(
            prediction["raw_response"]
        )

        if parsing_error is not None:
            print(
                f"{index}/{len(pilot_documents)} | "
                f"Document {document['document_id']} | Invalid JSON"
            )

            results.append({
                "document_id": document["document_id"],
                "valid_json": False,
                "precision": 0.0,
                "recall": 0.0,
                "f1": 0.0
            })

            continue

        parsed_output = grouped_output_to_entity_json(
            grouped_output
        )

        evaluation = evaluate_entity_predictions(
            parsed_output=parsed_output,
            document=document
        )

        results.append({
            "document_id": document["document_id"],
            "valid_json": True,
            "precision": evaluation["precision"],
            "recall": evaluation["recall"],
            "f1": evaluation["f1"]
        })

        print(
            f"{index}/{len(pilot_documents)} | "
            f"Document {document['document_id']} | "
            f"F1 = {evaluation['f1']:.4f}"
        )

    return results

In [ ]:
pilot_results = evaluate_pilot_documents(
    pilot_documents=pilot_documents,
    example_documents=few_shot_examples,
    example_outputs=few_shot_demo_outputs
)

1/5 | Document 16680561 | F1 = 0.8421
2/5 | Document 27840894 | F1 = 0.6441
3/5 | Document 16277682 | F1 = 0.5205
4/5 | Document 20195852 | F1 = 0.7660
5/5 | Document 21130517 | F1 = 0.5636


In [44]:
import pandas as pd


In [ ]:
pilot_summary = pd.DataFrame(pilot_results)

display(pilot_summary.round(4))

print()

print("Average Precision:",
      pilot_summary["precision"].mean())

print("Average Recall:",
      pilot_summary["recall"].mean())

print("Average F1:",
      pilot_summary["f1"].mean())

,document_id,valid_json,gold_entities,predicted_entities,tp,fp,fn,precision,recall,f1
0,16680561,True,28,29,24,5,4,0.8276,0.8571,0.8421
1,27840894,True,35,24,19,5,16,0.7917,0.5429,0.6441
2,16277682,True,38,35,19,16,19,0.5429,0.5000,0.5205
3,20195852,True,51,43,36,7,15,0.8372,0.7059,0.7660
4,21130517,True,43,67,31,36,12,0.4627,0.7209,0.5636



Average Precision: 0.6924011771820243
Average Recall: 0.6653625170998632
Average F1: 0.6672629630836837


## Stage 4.6: Few-Shot Development Set Evaluation

This stage applies the frozen few-shot NER pipeline to the remaining BioRED development documents. Existing generation, parsing, conversion, validation, mention recovery, and strict evaluation functions are reused without modification.

In [ ]:
# Stage 4.6A: Select Development Documents

pilot_ids = {
    document["document_id"]
    for document in pilot_documents
}

development_documents = [
    document
    for document in prepared_dev_documents
    if document["document_id"] not in pilot_ids
]

print("Development documents selected:", len(development_documents))

Development documents selected: 95


In [ ]:
# Stage 4.6B: Configure Development Evaluation Checkpoint

import os
import pickle

DEV_RESULTS_PATH = (
    "/content/drive/MyDrive/Capstone_Project/"
    "few_shot_development_ner_results.pkl"
)

if os.path.exists(DEV_RESULTS_PATH):
    with open(DEV_RESULTS_PATH, "rb") as file:
        development_results = pickle.load(file)

    print("Loaded existing results:", len(development_results))
else:
    development_results = []
    print("Starting a new development evaluation.")

Starting a new development evaluation.


In [ ]:
# Stage 4.6C: Run Development Evaluation with Periodic Checkpointing

completed_ids = {
    result["document_id"]
    for result in development_results
}

remaining_documents = [
    document
    for document in development_documents
    if document["document_id"] not in completed_ids
]

print("Already completed:", len(development_results))
print("Remaining documents:", len(remaining_documents))

for index, document in enumerate(remaining_documents, start=1):
    result = evaluate_pilot_documents(
        pilot_documents=[document],
        example_documents=few_shot_examples,
        example_outputs=few_shot_demo_outputs
    )[0]

    development_results.append(result)

    print(
        f"{len(development_results)}/"
        f"{len(development_documents)} | "
        f"Document: {document['document_id']} | "
        f"JSON: {result['valid_json']} | "
        f"F1: {result['f1']:.4f}"
    )

    if index % 10 == 0:
        with open(DEV_RESULTS_PATH, "wb") as file:
            pickle.dump(development_results, file)

        print("Checkpoint saved.")

# Final save after all remaining documents
with open(DEV_RESULTS_PATH, "wb") as file:
    pickle.dump(development_results, file)

print("Development evaluation complete.")
print("Total saved results:", len(development_results))

Already completed: 0
Remaining documents: 95
1/1 | Document 14510914 | F1 = 0.3125
1/95 | Document: 14510914 | JSON: True | F1: 0.3125
1/1 | Document 15096016 | F1 = 0.5000
2/95 | Document: 15096016 | JSON: True | F1: 0.5000
1/1 | Document 16152606 | F1 = 0.2857
3/95 | Document: 16152606 | JSON: True | F1: 0.2857
1/1 | Document 1671881 | F1 = 0.5455
4/95 | Document: 1671881 | JSON: True | F1: 0.5455
1/1 | Document 16867246 | F1 = 0.5316
5/95 | Document: 16867246 | JSON: True | F1: 0.5316
1/1 | Document 17549393 | F1 = 0.7234
6/95 | Document: 17549393 | JSON: True | F1: 0.7234
1/1 | Document 17962394 | F1 = 0.4762
7/95 | Document: 17962394 | JSON: True | F1: 0.4762
1/1 | Document 18046082 | F1 = 0.6061
8/95 | Document: 18046082 | JSON: True | F1: 0.6061
1/1 | Document 21054465 | F1 = 0.3692
9/95 | Document: 21054465 | JSON: True | F1: 0.3692
1/1 | Document 22711886 | F1 = 0.7207
10/95 | Document: 22711886 | JSON: True | F1: 0.7207
Checkpoint saved.
1/1 | Document 24036311 | F1 = 0.5862


In [48]:
development_summary = pd.DataFrame(development_results)

display(development_summary.round(4))

NameError: name 'development_results' is not defined

In [81]:
print("Development-set evaluation")
print("--------------------------")
print("Documents:", len(development_summary))
print("Valid JSON outputs:", development_summary["valid_json"].sum())
print("Invalid JSON outputs:", (~development_summary["valid_json"]).sum())

Development-set evaluation
--------------------------
Documents: 95
Valid JSON outputs: 91
Invalid JSON outputs: 4


In [82]:
print("Development-set few-shot NER evaluation")
print("--------------------------------------")
print("Documents:", len(development_summary))
print("Valid JSON outputs:", development_summary["valid_json"].sum())
print("Invalid JSON outputs:", (~development_summary["valid_json"]).sum())

print()

print(f"Average Precision: {development_summary['precision'].mean():.4f}")
print(f"Average Recall:    {development_summary['recall'].mean():.4f}")
print(f"Average F1-score:  {development_summary['f1'].mean():.4f}")

Development-set few-shot NER evaluation
--------------------------------------
Documents: 95
Valid JSON outputs: 91
Invalid JSON outputs: 4

Average Precision: 0.5964
Average Recall:    0.5313
Average F1-score:  0.5495


##Stage 4.7: Few-Shot Test Set Evaluation

In [40]:
# Stage 4.7A: Select Test Documents

test_documents = prepared_test_documents

print("Test documents:", len(test_documents))

Test documents: 100


In [41]:
# Stage 4.7B: Configure Test Evaluation

import os
import pickle

TEST_RESULTS_PATH = (
    "/content/drive/MyDrive/Capstone_Project/"
    "few_shot_test_ner_results.pkl"
)

if os.path.exists(TEST_RESULTS_PATH):
    with open(TEST_RESULTS_PATH, "rb") as file:
        test_results = pickle.load(file)

    print("Loaded existing results:", len(test_results))
else:
    test_results = []
    print("Starting new test evaluation.")

Loaded existing results: 50


In [42]:
# Stage 4.7C: Evaluate Test Documents

completed_ids = {
    result["document_id"]
    for result in test_results
}

remaining_documents = [
    document
    for document in test_documents
    if document["document_id"] not in completed_ids
]

print("Already completed:", len(test_results))
print("Remaining documents:", len(remaining_documents))

for index, document in enumerate(remaining_documents, start=1):

    result = evaluate_pilot_documents(
        pilot_documents=[document],
        example_documents=few_shot_examples,
        example_outputs=few_shot_demo_outputs
    )[0]

    test_results.append(result)

    print(
        f"{len(test_results)}/{len(test_documents)} | "
        f"Document: {document['document_id']} | "
        f"JSON: {result['valid_json']} | "
        f"F1: {result['f1']:.4f}"
    )

    if index % 10 == 0:
        with open(TEST_RESULTS_PATH, "wb") as file:
            pickle.dump(test_results, file)

        print("Checkpoint saved.")

with open(TEST_RESULTS_PATH, "wb") as file:
    pickle.dump(test_results, file)

print("\nTest evaluation complete.")

Already completed: 50
Remaining documents: 50
1/1 | Document 27798239 | F1 = 0.5846
51/100 | Document: 27798239 | JSON: True | F1: 0.5846
1/1 | Document 27993978 | F1 = 0.5517
52/100 | Document: 27993978 | JSON: True | F1: 0.5517
1/1 | Document 15111599 | F1 = 0.5600
53/100 | Document: 15111599 | JSON: True | F1: 0.5600
1/1 | Document 15820770 | F1 = 0.5116
54/100 | Document: 15820770 | JSON: True | F1: 0.5116
1/1 | Document 16186368 | F1 = 0.6000
55/100 | Document: 16186368 | JSON: True | F1: 0.6000
1/1 | Document 17511042 | F1 = 0.6316
56/100 | Document: 17511042 | JSON: True | F1: 0.6316
1/1 | Document 26110643 | Invalid JSON
57/100 | Document: 26110643 | JSON: False | F1: 0.0000
1/1 | Document 18179903 | F1 = 0.7042
58/100 | Document: 18179903 | JSON: True | F1: 0.7042
1/1 | Document 19353688 | F1 = 0.5185
59/100 | Document: 19353688 | JSON: True | F1: 0.5185
1/1 | Document 21126715 | F1 = 0.4186
60/100 | Document: 21126715 | JSON: True | F1: 0.4186
Checkpoint saved.
1/1 | Document

In [45]:
# Stage 4.7D: Summarise Test Results

test_summary = pd.DataFrame(test_results)

display(test_summary.round(4))

print()

print("Test-set few-shot NER evaluation")
print("--------------------------------")
print("Documents:", len(test_summary))
print("Valid JSON outputs:", test_summary["valid_json"].sum())
print("Invalid JSON outputs:", (~test_summary["valid_json"]).sum())

print()

print(f"Average Precision: {test_summary['precision'].mean():.4f}")
print(f"Average Recall:    {test_summary['recall'].mean():.4f}")
print(f"Average F1-score:  {test_summary['f1'].mean():.4f}")

,document_id,valid_json,precision,recall,f1
0,15485686,True,0.5000,0.2667,0.3478
1,16046395,True,0.8333,0.7143,0.7692
2,18457324,True,0.9091,0.5970,0.7207
3,1848636,True,0.5556,0.8621,0.6757
4,19394258,True,0.6667,0.6364,0.6512
...,...,...,...,...,...
95,19531695,True,0.7755,0.7755,0.7755
96,24114426,True,0.6471,0.6875,0.6667
97,24717468,True,0.9048,0.7308,0.8085
98,24739405,True,0.9333,0.9032,0.9180



Test-set few-shot NER evaluation
--------------------------------
Documents: 100
Valid JSON outputs: 97
Invalid JSON outputs: 3

Average Precision: 0.6484
Average Recall:    0.5445
Average F1-score:  0.5768


In [46]:
test_summary.to_csv(
    "/content/drive/MyDrive/Capstone_Project/few_shot_test_ner_results.csv",
    index=False
)

print("CSV saved successfully.")

CSV saved successfully.


In [49]:
# Reload saved development results

with open(
    "/content/drive/MyDrive/Capstone_Project/"
    "few_shot_development_ner_results.pkl",
    "rb"
) as file:
    development_results = pickle.load(file)

development_summary = pd.DataFrame(development_results)

print("Loaded development results:", len(development_summary))

Loaded development results: 95


In [50]:
development_summary.to_csv(
    "/content/drive/MyDrive/Capstone_Project/few_shot_development_ner_results.csv",
    index=False
)

print("Development CSV saved successfully.")

Development CSV saved successfully.


In [6]:
# Mount Google Drive

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [12]:
# Stage 4.8A: Save Recovered PubMedBERT NER Test Results

import json

pubmedbert_ner_results = {
    "precision": 0.8458007021,
    "recall": 0.8859971711,
    "f1": 0.8654324399,
    "accuracy": 0.9679572764
}

PUBMEDBERT_NER_PATH = (
    "/content/drive/MyDrive/Capstone_Project/"
    "pubmedbert_ner_test_results.json"
)

with open(PUBMEDBERT_NER_PATH, "w") as f:
    json.dump(pubmedbert_ner_results, f, indent=2)

print("PubMedBERT NER results saved.")

PubMedBERT NER results saved.


In [13]:
# Stage 4.8B: Load Saved NER Results

import json
import pandas as pd

with open(PUBMEDBERT_NER_PATH, "r") as f:
    pubmedbert_ner = json.load(f)

QWEN_NER_PATH = (
    "/content/drive/MyDrive/Capstone_Project/"
    "few_shot_test_ner_results.csv"
)

qwen_ner = pd.read_csv(QWEN_NER_PATH)

print("PubMedBERT NER results loaded.")
print("Qwen test documents:", len(qwen_ner))

PubMedBERT NER results loaded.
Qwen test documents: 100


In [14]:
# Stage 4.8C: Compute Qwen NER Test Metrics

qwen_metrics = {
    "precision": qwen_ner["precision"].mean(),
    "recall": qwen_ner["recall"].mean(),
    "f1": qwen_ner["f1"].mean(),
    "valid_json": int(qwen_ner["valid_json"].sum())
}

print(f"Precision:  {qwen_metrics['precision']:.4f}")
print(f"Recall:     {qwen_metrics['recall']:.4f}")
print(f"F1-score:   {qwen_metrics['f1']:.4f}")
print(f"Valid JSON: {qwen_metrics['valid_json']}/{len(qwen_ner)}")

Precision:  0.6484
Recall:     0.5445
F1-score:   0.5768
Valid JSON: 97/100


In [15]:
# Stage 4.8D: Compare PubMedBERT and Qwen NER

ner_comparison = pd.DataFrame([
    {
        "Model": "PubMedBERT (Fine-tuned)",
        "Precision": pubmedbert_ner["precision"],
        "Recall": pubmedbert_ner["recall"],
        "F1": pubmedbert_ner["f1"]
    },
    {
        "Model": "Qwen2.5-7B-Instruct (Few-shot)",
        "Precision": qwen_metrics["precision"],
        "Recall": qwen_metrics["recall"],
        "F1": qwen_metrics["f1"]
    }
])

ner_comparison.round(4)

,Model,Precision,Recall,F1
0,PubMedBERT (Fine-tuned),0.8458,0.8860,0.8654
1,Qwen2.5-7B-Instruct (Few-shot),0.6484,0.5445,0.5768


### Stage 4.8E: NER Model Comparison Summary

The fine-tuned PubMedBERT model achieved substantially better NER performance than Qwen2.5-7B-Instruct using few-shot prompting.

PubMedBERT achieved an F1-score of **0.8654**, compared with **0.5768** for Qwen. PubMedBERT also achieved higher precision (**0.8458 vs. 0.6484**) and recall (**0.8860 vs. 0.5445**).

These results indicate that supervised fine-tuning of a domain-specific transformer was more effective for BioRED named entity recognition than few-shot prompting of the general-purpose LLM in this experiment.

**Evaluation note:** PubMedBERT was evaluated using sequence-labelling metrics with `seqeval`, whereas the Qwen results were calculated using strict entity-level matching and averaged across test documents. Therefore, the scores provide an experimental comparison of the two pipelines but are not based on identical metric aggregation procedures.